# Ejercicio 3 — Jugadores de Copas Mundiales: ACP, t-SNE y UMAP
**Lead University · Minería de Datos · Tarea 5**

Comparación de ACP, t-SNE y UMAP sobre el dataset `Players1.csv`:  
595 jugadores de Copas Mundiales con estadísticas de juego, coloreados por posición táctica.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from scripts import DimReducer

print('Librerías cargadas.')

Librerías cargadas.


---
## a) Carga de datos

In [2]:
df = pd.read_csv('datos/Players1.csv')
print(f'Dimensiones: {df.shape}')
print(f'Columnas: {df.columns.tolist()}')
print(f'\nPosiciones:')
print(df['position'].value_counts())
df.head()

Dimensiones: (595, 8)
Columnas: ['surname', 'team', 'position', 'minutes', 'shots', 'passes', 'tackles', 'saves']

Posiciones:
position
midfielder    228
defender      188
forward       143
goalkeeper     36
Name: count, dtype: int64


,surname,team,position,minutes,shots,passes,tackles,saves
0,Abdoun,Algeria,midfielder,16,0,6,0,0
1,Belhadj,Algeria,defender,270,1,146,8,0
2,Boudebouz,Algeria,midfielder,74,3,28,1,0
3,Bougherra,Algeria,defender,270,1,89,11,0
4,Chaouchi,Algeria,goalkeeper,90,0,17,0,2


In [3]:
df[['minutes', 'shots', 'passes', 'tackles', 'saves']].describe().round(2)

,minutes,shots,passes,tackles,saves
count,595.00,595.00,595.00,595.00,595.00
mean,208.86,2.30,84.52,4.19,0.67
std,145.43,3.35,77.31,4.65,2.96
min,1.00,0.00,0.00,0.00,0.00
25%,88.00,0.00,29.00,1.00,0.00
50%,191.00,1.00,61.00,3.00,0.00
75%,270.00,3.00,115.50,6.00,0.00
max,570.00,27.00,563.00,34.00,20.00


**Contexto del dataset:**  
Las estadísticas (minutos, disparos, pases, tackles, atajadas) reflejan el rol táctico de cada jugador.
Se usa `position` solo para colorear — no entra al modelo. Si los métodos capturan la estructura,
deben separar las posiciones tácticas sin haberla usado como variable de entrada.

---
## b) ACP, t-SNE y UMAP con 3 componentes

### Exploración de n_neighbors para UMAP

In [4]:
# Quitar columnas no numéricas que no son la de color
df_model = df.drop(columns=['surname', 'team'])

dr = DimReducer(df_model, color_col='position', seed=42)

dr.explore_umap_neighbors(neighbors_list=[5, 15, 20, 50])

**Selección de n_neighbors:**  
Se probaron los valores 5, 15, 20 y 50. Con n_neighbors=5 los datos se fragmentan en
micro-grupos inconexos — los porteros (verde) aparecen dispersos en varios mini-clústeres
en lugar de un grupo cohesionado. Con n_neighbors=15 la estructura mejora y los porteros
quedan aislados, pero el resto de posiciones forma una cinta continua sin separación clara.
Con n_neighbors=20 los porteros quedan bien aislados a la izquierda como grupo compacto,
y se empieza a apreciar cierta diferenciación entre delanteros y el bloque defensa/mediocampistas.
Con n_neighbors=50 todo colapsa en una diagonal continua donde las posiciones se mezclan
completamente. **Se selecciona n_neighbors = 20** como el valor que produce la separación
más clara entre posiciones tácticas.

In [5]:
dr.fit(n_components=3, tsne_perplexity=30, umap_n_neighbors=20)

Ajuste completado — n_components=3, perplexity=30, n_neighbors=20


---
## c) Gráficos en las dos primeras componentes

In [6]:
dr.plot_mapa_interactivo(dr.coords_pca, title='Jugadores Mundiales — ACP')
dr.plot_mapa_interactivo(dr.coords_tsne, title='Jugadores Mundiales — t-SNE')
dr.plot_mapa_interactivo(dr.coords_umap, title='Jugadores Mundiales — UMAP')

In [7]:
dr.plot_comparacion()

---
## d) Comparación e interpretación

**Nota sobre la distribución de posiciones:** El dataset está desbalanceado: mediocampistas
(228), defensas (188) y delanteros (143) son grupos grandes, mientras que los porteros son
solo 36 de 595 jugadores (6%). Esto significa que la separación visual de porteros en los
mapas se debe tanto a su perfil estadístico atípico como a su escasez relativa.

**ACP — Proyección lineal:**  
Logra separar a los **porteros** del resto: su perfil radicalmente distinto (muchos `saves`,
casi ningún `shot` ni `pass`) los coloca claramente en la parte superior del plano, bien
separados del resto. Sin embargo, defensas, mediocampistas y delanteros quedan completamente
aplanados en una línea horizontal cerca de Y=0, sin ninguna separación entre ellos. El ACP
solo captura la diferencia extrema de los porteros, pero es incapaz de distinguir entre los
tres roles de campo.

**t-SNE — Preservación de vecindades:**  
Los porteros forman un clúster compacto y bien aislado en la parte superior. El resto de
jugadores forma una masa densa donde defensas, mediocampistas y delanteros se mezclan
considerablemente. Hay algo de estructura interna pero no es suficiente para separar los
tres roles de campo de forma clara.

**UMAP — Mejor representación global:**  
Ofrece la separación más informativa. Los porteros quedan aislados arriba-izquierda.
El cuerpo principal forma una estructura curva donde los delanteros (naranja) tienden a
concentrarse en la zona inferior-derecha, diferenciándose del bloque de defensas y
mediocampistas que ocupan la zona superior-derecha. Es el único método que sugiere
cierta diferenciación entre los roles de campo, aunque el solapamiento entre defensas
y mediocampistas persiste — esperablemente, dado que sus estadísticas se diferencian
en grado más que en tipo.

**Conclusión:**  
UMAP identifica mejor los patrones según estadísticas de juego. t-SNE ocupa el segundo lugar.
ACP solo es útil para detectar porteros como grupo atípico extremo, pero proyecta todos los
demás jugadores en una sola dimensión sin estructura. El resultado más relevante es que
**esta separación emergió únicamente de las cinco variables numéricas** (minutos, disparos,
pases, tackles, atajadas) — la variable `position` no entró al modelo en ningún momento,
solo se usó para colorear los puntos. Que los métodos hayan recuperado al menos la separación
de porteros, y que UMAP sugiera diferenciación adicional entre roles, demuestra que las
estadísticas de juego contienen señal táctica real.